# ChefBot AI - addestramento del classificatore

In questo notebook preparo un classificatore per riconoscere 12 piatti del dataset Food101.

Ho scelto MobileNetV2 perché è una rete abbastanza leggera e si presta bene al transfer learning. Prima alleno soltanto i nuovi livelli finali, poi provo un breve fine-tuning su una parte della rete pre-addestrata.

## 1. Librerie e impostazioni

Per rendere le prove ripetibili imposto un seed. Le immagini vengono ridimensionate a 224 x 224 pixel, che è la dimensione prevista da MobileNetV2.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import tensorflow as tf
import tensorflow_datasets as tfds

from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

tf.keras.utils.set_random_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU disponibili:", tf.config.list_physical_devices("GPU"))

## 2. Caricamento di Food101

Food101 contiene 101 categorie. Per questo progetto ne utilizzo 12, così il training rimane gestibile anche con risorse limitate.

In [ ]:
SELECTED_CLASSES = [
    "bruschetta",
    "caprese_salad",
    "cheesecake",
    "greek_salad",
    "lasagna",
    "paella",
    "panna_cotta",
    "pizza",
    "ravioli",
    "spaghetti_bolognese",
    "spaghetti_carbonara",
    "tiramisu",
]

(ds_train_full, ds_test_full), info = tfds.load(
    "food101",
    split=["train", "validation"],
    as_supervised=True,
    with_info=True,
)

food101_names = info.features["label"].names
selected_ids = [food101_names.index(name) for name in SELECTED_CLASSES]

print("Immagini totali di training:", info.splits["train"].num_examples)
print("Immagini totali di test:", info.splits["validation"].num_examples)
print("Classi scelte:", len(SELECTED_CLASSES))

### Filtro e nuova codifica delle etichette

Le etichette originali sono comprese tra 0 e 100. Dopo il filtro le rimappo da 0 a 11, nello stesso ordine usato dall'applicazione.

In [ ]:
selected_ids_tensor = tf.constant(selected_ids, dtype=tf.int64)

keys = tf.constant(selected_ids, dtype=tf.int64)
values = tf.range(len(SELECTED_CLASSES), dtype=tf.int64)
label_table = tf.lookup.StaticHashTable(
    tf.lookup.KeyValueTensorInitializer(keys, values),
    default_value=-1,
)

def keep_selected(image, label):
    return tf.reduce_any(tf.equal(label, selected_ids_tensor))

def remap_label(image, label):
    return image, label_table.lookup(label)

train_ds = ds_train_full.filter(keep_selected).map(
    remap_label,
    num_parallel_calls=tf.data.AUTOTUNE,
)
test_ds = ds_test_full.filter(keep_selected).map(
    remap_label,
    num_parallel_calls=tf.data.AUTOTUNE,
)

## 3. Controllo di alcune immagini

Prima del training visualizzo un piccolo campione per verificare che immagini ed etichette corrispondano.

In [ ]:
plt.figure(figsize=(12, 8))

for index, (image, label) in enumerate(train_ds.take(12)):
    ax = plt.subplot(3, 4, index + 1)
    plt.imshow(image)
    plt.title(SELECTED_CLASSES[int(label.numpy())].replace("_", " "))
    plt.axis("off")

plt.tight_layout()
plt.show()

## 4. Preparazione dei dataset

Uso una piccola data augmentation solo sul training set. Non applico trasformazioni troppo forti perché potrebbero cambiare l'aspetto caratteristico del piatto.

In [ ]:
def resize_image(image, label):
    image = tf.image.resize(image, IMG_SIZE)
    return image, label

train_ds = (
    train_ds
    .map(resize_image, num_parallel_calls=tf.data.AUTOTUNE)
    .shuffle(3000, seed=SEED)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

test_ds = (
    test_ds
    .map(resize_image, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

data_augmentation = tf.keras.Sequential(
    [
        tf.keras.layers.RandomFlip("horizontal"),
        tf.keras.layers.RandomRotation(0.08),
        tf.keras.layers.RandomZoom(0.10),
    ],
    name="data_augmentation",
)

## 5. Transfer learning con MobileNetV2

Carico i pesi ImageNet ed escludo il classificatore originale. Nella prima fase la base convoluzionale resta congelata.

In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights="imagenet",
)
base_model.trainable = False

inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.25)(x)
outputs = tf.keras.layers.Dense(
    len(SELECTED_CLASSES),
    activation="softmax",
)(x)

model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=2,
    ),
]

history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=8,
    callbacks=callbacks,
)

## 6. Andamento del primo training

Confronto accuracy e loss per controllare se il modello sta migliorando e se compare overfitting.

In [ ]:
def plot_history(training_history, title):
    history_data = training_history.history
    epochs = range(1, len(history_data["loss"]) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(epochs, history_data["accuracy"], label="training")
    axes[0].plot(epochs, history_data["val_accuracy"], label="validation")
    axes[0].set_title(f"{title} - accuracy")
    axes[0].set_xlabel("Epoca")
    axes[0].legend()

    axes[1].plot(epochs, history_data["loss"], label="training")
    axes[1].plot(epochs, history_data["val_loss"], label="validation")
    axes[1].set_title(f"{title} - loss")
    axes[1].set_xlabel("Epoca")
    axes[1].legend()

    plt.tight_layout()
    plt.show()

plot_history(history, "Transfer learning")

## 7. Fine-tuning

Sblocco soltanto gli ultimi 20 livelli della base. Uso un learning rate basso per non modificare troppo velocemente i pesi già appresi su ImageNet.

In [ ]:
base_model.trainable = True

for layer in base_model.layers[:-20]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

fine_tune_history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=5,
    callbacks=callbacks,
)

In [ ]:
plot_history(fine_tune_history, "Fine-tuning")

test_loss, test_accuracy = model.evaluate(test_ds, verbose=0)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")

## 8. Matrice di confusione

La matrice di confusione è utile per capire quali piatti vengono scambiati più spesso. In particolare mi aspetto qualche difficoltà tra piatti di pasta o dessert con colori e consistenze simili.

In [ ]:
true_labels = []
predicted_labels = []

for images, labels in test_ds:
    probabilities = model.predict(images, verbose=0)
    predictions = np.argmax(probabilities, axis=1)

    true_labels.extend(labels.numpy())
    predicted_labels.extend(predictions)

matrix = confusion_matrix(true_labels, predicted_labels)

plt.figure(figsize=(12, 10))
sns.heatmap(
    matrix,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=SELECTED_CLASSES,
    yticklabels=SELECTED_CLASSES,
)
plt.xlabel("Classe prevista")
plt.ylabel("Classe reale")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

print(
    classification_report(
        true_labels,
        predicted_labels,
        target_names=SELECTED_CLASSES,
        digits=3,
    )
)

## 9. Salvataggio

Salvo sia il modello sia l'ordine delle classi. Entrambi servono all'app Streamlit per associare la previsione alla scheda corretta.

In [ ]:
models_dir = Path("../models")
models_dir.mkdir(exist_ok=True)

model.save(models_dir / "chefbot_mobilenet.keras")

with (models_dir / "class_names.json").open("w", encoding="utf-8") as file:
    json.dump(SELECTED_CLASSES, file, indent=2)

print("Modello e classi salvati nella cartella models.")

## Considerazioni finali

Il risultato va valutato insieme alla matrice di confusione e non soltanto attraverso l'accuracy complessiva. Food101 contiene fotografie molto diverse per inquadratura, luce e presentazione del piatto, quindi alcuni errori sono prevedibili.

Come possibile sviluppo futuro si potrebbero aumentare le classi, aggiungere più data augmentation e analizzare separatamente le immagini sulle quali il modello mostra poca sicurezza.